# California Housing Prices: Regresión con PyTorch y Weights & Biases

En esta notebook, implementaremos un modelo de regresión para predecir valores medianos de casas basado en un conjunto de datos de características demográficas y geográficas. Además, presentaremos los aspectos básicos del entrenamiento de modelos en PyTorch y utilizaremos la herramienta Weights & Biases (W&B) para el seguimiento y visualización de los experimentos.

## Introducción

### Objetivos

1. **Implementar un modelo de regresión** utilizando PyTorch.
2. **Configurar y utilizar W&B** para el seguimiento de métricas y visualización de resultados.

### Contenido

1. Configuración de bibliotecas y semillas para reproducibilidad.
2. Carga y exploración del dataset California Housing de scikit-learn.
3. Preparación de datos y división en conjuntos de entrenamiento y prueba.
4. Definición y entrenamiento de un modelo de regresión en PyTorch.
5. Integración de W&B para el seguimiento de experimentos y visualización de métricas.

### Sobre el conjunto de datos

Los datos se refieren a las casas que se encuentran en un determinado distrito de California y algunas estadísticas resumidas sobre ellas basadas en los datos del censo de 1990. La versión que distribuye scikit-learn ya está limpia (sin valores faltantes ni columnas categóricas), por lo que el único preprocesamiento que haremos es la normalización de las variables. El objetivo es predecir el valor medio de las casas para los distritos de California, por lo que se trata de un problema de regresión.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from torchinfo import summary

from sklearn.datasets import fetch_california_housing
import matplotlib.pyplot as plt

In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 34

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True  # cuDNN tiene varias implementaciones de cada operación (convoluciones, etc.); esto fuerza las que dan siempre el mismo resultado, a costa de velocidad

In [ ]:
import sys

# definimos el dispositivo que vamos a usar
DEVICE = "cpu"  # por defecto, usamos la CPU
if torch.cuda.is_available():
    DEVICE = "cuda"  # si hay GPU, usamos la GPU
elif torch.backends.mps.is_available():
    DEVICE = "mps"  # si no hay GPU, pero hay MPS, usamos MPS
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = "xpu"  # si no hay GPU, pero hay XPU, usamos XPU

print(f"Usando {DEVICE}")

# num_workers > 0 hace que el DataLoader cargue los batches en procesos hijos, en paralelo.
# En Linux los hijos se crean con fork (copia del proceso actual, con el Dataset ya en memoria).
# En Windows y macOS se crean con spawn: un intérprete nuevo que debe reconstruir el Dataset
# importándolo por nombre, y las clases definidas en un notebook no son importables -> errores de pickle.
# El valor 4 es arbitrario: depende de los núcleos y del costo de cargar cada muestra.
# Acá los datos ya están en memoria, así que 0 workers rinde casi igual; importa con imágenes en disco.
NUM_WORKERS = 0  # Win y MacOS pueden tener problemas con múltiples workers
if sys.platform == "linux":
    NUM_WORKERS = 4  # numero de workers para cargar los datos (depende de cada caso)

print(f"Usando {NUM_WORKERS}")

In [ ]:
BATCH_SIZE = 1024  # tamaño del batch

## Carga de datos + Exploración

In [ ]:
california_housing = fetch_california_housing(as_frame=True)  # as_frame=True: devuelve DataFrames de pandas en lugar de arrays de numpy
target_column_name = california_housing.target_names[0]  # 'MedHouseVal'
print(california_housing.DESCR)

In [ ]:
df = california_housing.frame  # Convertimos el dataset en un DataFrame de pandas
df.head()  # Mostramos las primeras filas del DataFrame

In [ ]:
df.describe()  # Mostramos un resumen de las estadísticas del DataFrame

In [ ]:
# Chequear si hay valores nulos
df.isnull().sum()

Afortunadamente, el conjunto de datos no tiene valores faltantes, por lo que podemos ignorar la imputación de datos.

In [ ]:
# Crear la matriz de correlación
correlation_matrix = df.corr()

# Mostrar la matriz de correlación como un heatmap
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(im, ax=ax)

columns = correlation_matrix.columns
ax.set_xticks(range(len(columns)), labels=columns, rotation=45, ha="right")
ax.set_yticks(range(len(columns)), labels=columns)

# anotamos cada celda con el valor de la correlación
for i in range(len(columns)):
    for j in range(len(columns)):
        ax.text(j, i, f"{correlation_matrix.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

ax.set_title("Matriz de correlación")
plt.tight_layout()
plt.show()

La única variable con una correlación fuerte con `MedHouseVal` es `MedInc` (ingreso medio, ~0.69), lo cual suena lógico. El resto de las variables tiene una correlación débil con el target (por ejemplo `HouseAge` ~0.11 y `AveRooms` ~0.15). También vemos que `AveRooms` y `AveBedrms` están muy correlacionadas entre sí (~0.85), es decir, aportan información redundante. Podríamos descartar algunas variables que no aportan mucho valor al modelo, pero por simplicidad, las mantendremos todas.

## Preprocesamiento de datos

Para el preprocesamiento de datos nos vamos a limitar a la **estandarización** de las variables de entrada: a cada columna le restamos su media y la dividimos por su desvío estándar (z-score), de modo que todas queden centradas en 0 con desvío 1. Esto es una técnica común en el aprendizaje automático para mejorar la convergencia y la estabilidad del entrenamiento. El target `MedHouseVal` lo dejamos sin escalar.

Antes de normalizar dividimos el conjunto de datos en tres partes: entrenamiento, validación y prueba. La división se realiza de forma aleatoria, utilizando el 80% de los datos para entrenamiento, el 10% para validación y el 10% restante para pruebas.

> **Importante:** la media y el desvío se calculan **únicamente con el conjunto de entrenamiento** y luego se aplican a validación y test. Si los calculáramos con todo el dataset estaríamos filtrando información de los datos de evaluación hacia el entrenamiento (*data leakage*).

> **¿Por qué estandarizar?** La referencia clásica es LeCun, Bottou, Orr y Müller, *Efficient BackProp* (1998), sección 4.3: http://yann.lecun.com/exdb/publis/pdf/lecun-98b.pdf. Si las entradas tienen escalas muy distintas (por ejemplo `Population` en miles y `AveBedrms` cerca de 1), un mismo learning rate resulta demasiado grande para unos pesos y demasiado chico para otros. Con media 0 y desvío 1 en todas las variables, un único learning rate sirve para todas las direcciones y el entrenamiento converge más rápido.

In [ ]:
total_size = len(df)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = (
    total_size - train_size - val_size
)  # nos aseguramos de que el tamaño del conjunto de test sea correcto

# permutación aleatoria (reproducible) de los índices
# usamos un Generator propio para que el split no dependa de cuántos números aleatorios se consumieron antes
generator = torch.Generator().manual_seed(SEED)
perm = torch.randperm(total_size, generator=generator).tolist()
train_idx = perm[:train_size]
val_idx = perm[train_size : train_size + val_size]
test_idx = perm[train_size + val_size :]

feature_columns = [c for c in df.columns if c != target_column_name]

# estadísticos calculados SOLO con los datos de entrenamiento
train_mean = df.loc[train_idx, feature_columns].mean()
train_std = df.loc[train_idx, feature_columns].std()


def normalize(frame):
    """Estandariza las columnas de entrada usando la media y el desvío de entrenamiento."""
    out = frame.copy()
    out[feature_columns] = (frame[feature_columns] - train_mean) / train_std
    return out


train_df = normalize(df.loc[train_idx])
val_df = normalize(df.loc[val_idx])
test_df = normalize(df.loc[test_idx])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
train_df.describe()  # Mostramos un resumen de las estadísticas del conjunto de entrenamiento normalizado

## Dataset y DataLoader

**Dataset**

En PyTorch, un `Dataset` es una clase que se encarga de cargar y preparar los datos para su posterior uso en el entrenamiento de modelos. Este objeto almacena los datos y proporciona un método para acceder a ellos de manera eficiente y estructurada. Generalmente, `Dataset` se personaliza según el tipo de datos que se maneja, por ejemplo, imágenes, texto o series temporales. Su implementación básica requiere definir los métodos `__len__` para devolver el tamaño del dataset y `__getitem__` para acceder a un elemento específico.

Más información sobre `Dataset` en la documentación oficial de PyTorch: https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset

**DataLoader**

El `DataLoader` es una clase que se utiliza para envolver un objeto `Dataset` y proporciona un acceso fácil a los datos en lotes durante el entrenamiento, además de otras funcionalidades como el barajado (shuffling) de datos y la carga paralela utilizando múltiples subprocesos. Esto es crucial para el entrenamiento eficiente de modelos, especialmente con grandes volúmenes de datos, ya que gestiona el uso de memoria y mejora el rendimiento computacional.

Más información sobre `DataLoader` en la documentación oficial de PyTorch: https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader

**Objetivo y Uso**

El uso conjunto de `Dataset` y `DataLoader` en PyTorch es fundamental para el flujo de trabajo de entrenamiento de modelos de aprendizaje automático. `Dataset` se encarga de la estructura y accesibilidad de los datos, mientras que `DataLoader` optimiza el proceso de iteración sobre los datos en lotes y facilita operaciones como la mezcla y la carga paralela. Esto permite que el entrenamiento de modelos sea más escalable y eficiente.


### Datasets

In [ ]:
class CaliforniaHousingDataset(Dataset):
    def __init__(self, dataframe, target_column):
        # TODO: guardar las features (todas las columnas menos target_column) y el target.
        #       Conviene pasarlos a numpy con .to_numpy(): indexar un array es más rápido que un DataFrame

    def __len__(self):
        # TODO: devolver la cantidad de muestras (filas)

    def __getitem__(self, idx):
        # TODO: devolver la muestra idx como una tupla (x, y) de tensores float32.
        #       x: shape (8,). y: shape (1,) para que coincida con la salida del modelo (batch, 1);
        #       si y queda con shape () MSELoss no falla, pero calcula mal la pérdida


train_dataset = CaliforniaHousingDataset(train_df, target_column_name)
val_dataset = CaliforniaHousingDataset(val_df, target_column_name)
test_dataset = CaliforniaHousingDataset(test_df, target_column_name)

print(len(train_dataset), len(val_dataset), len(test_dataset))

### DataLoaders

Definimos los dataloaders para cada conjunto de datos, estos son los que se encargan de cargar los datos en lotes durante el entrenamiento y la evaluación del modelo.

In [ ]:
def get_data_loaders(batch_size, num_workers):

    # TODO: crear un DataLoader por conjunto con batch_size y num_workers.
    #       shuffle=True solo en train: en val y test el orden no importa y así la evaluación es reproducible
    train_loader = # TODO

    val_loader = # TODO

    test_loader = # TODO

    return train_loader, val_loader, test_loader

In [ ]:
train_loader, val_loader, test_loader = get_data_loaders(
    BATCH_SIZE, NUM_WORKERS
)  # obtenemos los dataloaders

# probamos un batch del DataLoader
x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape, y_batch.shape)

In [ ]:
# podemos recorrer los batches con un bucle
for x_batch, y_batch in train_loader:
    print(x_batch.shape, y_batch.shape)
    break  # solo mostramos el primer batch

## Modelo

A continuación, definimos un perceptrón multicapa (MLP) para regresión utilizando PyTorch: capas lineales con activación ReLU y una única salida (el valor a predecir). 

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()  # obligatorio: inicializa nn.Module para que registre capas y parámetros
        # TODO: definir las capas como atributos (self.fc1 = nn.Linear(...), ...).
        #       Sugerencia: input_size -> 64 -> 512 -> 1. La última capa tiene 1 salida (el valor a predecir) y no lleva activación

    def forward(self, x):
        # TODO: aplicar las capas en orden con ReLU (F.relu) entre ellas y devolver la salida, shape (batch, 1)
        pass


input_len = len(feature_columns)  # cantidad de variables de entrada

# torchinfo necesita el shape de la entrada (batch, features) para calcular los shapes intermedios y la cantidad de parámetros
summary(MLP(input_len), input_size=(BATCH_SIZE, input_len))

## Entrenamiento

Vamos a definir una función de entrenamiento que se encargará de iterar sobre los datos de entrenamiento, calcular las predicciones del modelo, calcular la pérdida y actualizar los pesos del modelo utilizando el optimizador.

También definimos una función de evaluación que se encargará de calcular la pérdida y otras métricas de evaluación en el conjunto de validación.

In [ ]:
def evaluate(model, criterion, data_loader):
    """
    Evalúa el modelo en los datos proporcionados y calcula la pérdida promedio.

    Args:
        model (torch.nn.Module): El modelo que se va a evaluar.
        criterion (torch.nn.Module): La función de pérdida que se utilizará para calcular la pérdida.
        data_loader (torch.utils.data.DataLoader): DataLoader que proporciona los datos de evaluación.

    Returns:
        float: La pérdida promedio en el conjunto de datos de evaluación.

    """
    avg_loss = 0
    # TODO:
    #   1. poner el modelo en modo evaluación y desactivar el cálculo de gradientes
    #   2. recorrer los batches, calcular la pérdida de cada uno y acumularla
    #   3. devolver la pérdida promedio por batch
    return avg_loss


def train(
    model,
    optimizer,
    criterion,
    train_loader,
    val_loader,
    epochs=10,
    log_fn=None,
    log_every=1,
):
    """
    Entrena el modelo utilizando el optimizador y la función de pérdida proporcionados.

    Args:
        model (torch.nn.Module): El modelo que se va a entrenar.
        optimizer (torch.optim.Optimizer): El optimizador que se utilizará para actualizar los pesos del modelo.
        criterion (torch.nn.Module): La función de pérdida que se utilizará para calcular la pérdida.
        train_loader (torch.utils.data.DataLoader): DataLoader que proporciona los datos de entrenamiento.
        val_loader (torch.utils.data.DataLoader): DataLoader que proporciona los datos de validación.
        epochs (int): Número de épocas de entrenamiento (default: 10).
        log_fn (function): Función que se llamará después de cada log_every épocas con los argumentos (epoch, train_loss, val_loss) (default: None).
        log_every (int): Número de épocas entre cada llamada a log_fn (default: 1).

    Returns:
        Tuple[List[float], List[float]]: Una tupla con dos listas, la primera con el error de entrenamiento de cada época y la segunda con el error de validación de cada época.

    """
    epoch_train_errors = []  # colectamos el error de training para posterior analisis
    epoch_val_errors = []  # colectamos el error de validacion para posterior analisis

    # TODO: por cada época:
    #   1. poner el modelo en modo entrenamiento
    #   2. por cada batch: limpiar gradientes, forward, pérdida, backward, paso del optimizador
    #   3. calcular la pérdida promedio de la época y evaluar en validación
    #   4. guardar ambas pérdidas en las listas
    #   5. llamar a log_fn cada log_every épocas (si se pasó una)

    return epoch_train_errors, epoch_val_errors

Definimos hiperparámetros como el número de épocas, la tasa de aprendizaje y la función de pérdida.

In [ ]:
# Definimos los hiperparámetros
LR = 0.01
CRITERION = nn.MSELoss()
EPOCHS = 100

In [ ]:
def print_log(epoch, train_loss, val_loss):
    print(
        f"Epoch: {epoch + 1:03d}/{EPOCHS:03d} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f}"
    )


model = MLP(input_len).to(DEVICE)
optimizer = optim.SGD(model.parameters(), lr=LR)

epoch_train_errors, epoch_val_errors = train(
    model, optimizer, CRITERION, train_loader, val_loader, EPOCHS, print_log, 5
)  # log_fn=print_log, log_every=5: imprime cada 5 épocas

### Loss durante el entrenamiento

Es importante tener en cuenta que la pérdida es una medida de cuán bien se está desempeñando el modelo en el conjunto de entrenamiento. Sin embargo, la pérdida por sí sola no proporciona una visión completa del rendimiento del modelo. Por lo tanto, es fundamental evaluar el modelo en un conjunto de validación independiente para obtener una idea más precisa de su capacidad para generalizar a datos no vistos.

In [ ]:
def plot_training(train_errors, val_errors):
    # Graficar los errores
    plt.figure(figsize=(10, 5))  # Define el tamaño de la figura
    plt.plot(train_errors, label="Train Loss")  # Grafica la pérdida de entrenamiento
    plt.plot(val_errors, label="Validation Loss")  # Grafica la pérdida de validación
    plt.title("Training and Validation Loss")  # Título del gráfico
    plt.xlabel("Epochs")  # Etiqueta del eje X
    plt.ylabel("Loss")  # Etiqueta del eje Y
    plt.legend()  # Añade una leyenda
    plt.grid(True)  # Añade una cuadrícula para facilitar la visualización
    plt.show()  # Muestra el gráfico


plot_training(epoch_train_errors, epoch_val_errors)  # graficamos los errores

plot_training(
    epoch_train_errors[5:], epoch_val_errors[5:]
)  # graficamos los errores a partir de la epoca 5 (para ver mejor)

## Weights & Biases (W&B)

Weights & Biases es una plataforma que facilita el seguimiento, visualización y colaboración en experimentos de aprendizaje automático. Permite registrar métricas, gráficos y modelos de manera centralizada para facilitar el análisis y la comprensión de los resultados obtenidos durante el entrenamiento de modelos.

**Funcionalidades Principales**

1. **Seguimiento de Experimentos:** Permite registrar métricas clave como pérdida y precisión a lo largo del entrenamiento para cada experimento.

2. **Visualización Interactiva:** Proporciona gráficos dinámicos para explorar la evolución de métricas y comparar diferentes experimentos de manera intuitiva.

3. **Colaboración y Reproducibilidad:** Facilita compartir resultados con colegas y mantener un registro detallado de todos los experimentos realizados.

> Se requiere una cuenta de W&B, asi como crear un team/equipo y un proyecto para poder utilizar la plataforma. Para más información, visite: https://wandb.ai/site

In [ ]:
import wandb

wandb.login()  # pide la API key la primera vez (Settings > API keys en wandb.ai) y la guarda localmente

### Sweeps en Weights & Biases

Uno de los aspectos destacados de Weights & Biases son los "sweeps". Estos permiten explorar múltiples combinaciones de hiperparámetros de manera automática, registrando y comparando los resultados de cada configuración. Algunas características de los sweeps incluyen:

- **Automatización de Experimentos:** Ejecución paralela de múltiples configuraciones de hiperparámetros para optimizar el rendimiento del modelo.

- **Análisis de Resultados:** Visualización automática de los resultados de cada configuración para identificar la mejor combinación de hiperparámetros.

- **Ajuste Eficiente:** Permite ajustar rápidamente los hiperparámetros sin necesidad de intervención manual intensiva.

Más información sobre Weights & Biases en la documentación oficial: https://docs.wandb.ai/guides/sweeps

In [ ]:
WANDB_TEAM_NAME = "[WANDB_TEAM_NAME]"
WANDB_PROJECT = "[WANDB_PROJECT]"

# La configuración del sweep: los parámetros que queremos optimizar
sweep_config = {
    "name": "sweep-dreams-are-made-of-this",
    "method": "random",  # estrategia de búsqueda: random, grid o bayes
    "metric": {"name": "val_loss", "goal": "minimize"},  # "name" debe coincidir con la clave que se loggea en wandb.log
    "parameters": {
        "learning_rate": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e-1},
        "optimizer": {"values": ["adam", "sgd"]},
        "batch_size": {"values": [32, 256, 1024, 4096]},
    },
}

# Crea un nuevo sweep en W&B y devuelve su id (podríamos usar un sweep existente si ya lo hubiéramos creado)
sweep_id = wandb.sweep(sweep_config, entity=WANDB_TEAM_NAME, project=WANDB_PROJECT)

### Función Run

La función `run` de W&B se utiliza para iniciar un experimento y registrar métricas clave, hiperparámetros y otros detalles relevantes. Esta función se utiliza para rastrear y visualizar los resultados de los experimentos en la plataforma W&B.

In [ ]:
def wand_log(epoch, train_loss, val_loss):
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})


def sweep_run():
    """
    Función que se ejecutará en cada run del sweep.
    """
    # inicializar un nuevo run; el agente del sweep le inyecta los hiperparámetros elegidos
    wandb.init()
    # leer la configuración del run (los valores que el sweep asignó a esta corrida)
    config = wandb.config
    run_learning_rate = config.learning_rate
    run_optimizer = config.optimizer
    run_batch_size = config.batch_size

    # TODO:
    #   1. torch.manual_seed(SEED): todos los runs parten de la misma inicialización y la comparación es justa
    #   2. crear los dataloaders con get_data_loaders(run_batch_size, NUM_WORKERS) (el de test no se usa acá)
    #   3. crear el modelo en DEVICE y el optimizador según run_optimizer ("adam" -> optim.Adam, "sgd" -> optim.SGD)
    #      con lr=run_learning_rate
    #   4. entrenar con train(..., EPOCHS, log_fn=wand_log, log_every=1) para que W&B reciba val_loss en cada época
    #   5. guardar los pesos con torch.save(model.state_dict(), "model.pth") y subirlos con wandb.save("model.pth")
    #   6. wandb.finish() para cerrar el run antes de que el agente lance el siguiente

### Corremos el Sweep

Para ejecutar un sweep, creamos un agente que le pide a W&B un conjunto de hiperparámetros para cada experimento. Luego, ejecutamos cada experimento con esos hiperparámetros y registramos los resultados en W&B.

Es importante que podemos ejecutar en varios entornos, incluso al mismo tiempo, para explorar diferentes configuraciones de hiperparámetros y comparar los resultados.

In [ ]:
# el agente pide configuraciones al sweep y ejecuta sweep_run con cada una; count = cantidad de runs de este agente
wandb.agent(sweep_id, function=sweep_run, count=10)

## Mejor Modelo

Una vez que hemos ejecutado el sweep y registrado los resultados en W&B, podemos identificar el mejor modelo basado en las métricas de evaluación. 

In [ ]:
api = wandb.Api()  # cliente para consultar runs y sweeps ya finalizados (a diferencia de wandb.init, que crea runs)

# nos traemos el sweep (objeto) para analizar los resultados
sweep = api.sweep(f"{WANDB_TEAM_NAME}/{WANDB_PROJECT}/{sweep_id}")

# obtenemos el mejor run según la métrica y el goal definidos en sweep_config
best_run = sweep.best_run()

# imprimimos el mejor run
print(f"Best run {best_run.name} with {best_run.summary['val_loss']}")

# descargamos el modelo del mejor run
best_run.file("model.pth").download(replace=True)  # replace=True: sobrescribe si ya existe un model.pth local

## Evaluación final

Finalmente, evaluamos el mejor modelo en el conjunto de prueba. Creamos una instancia nueva del modelo y cargamos los pesos descargados de W&B.

> Si el sweep también explora la arquitectura (cantidad de capas, neuronas, activación), el modelo debe reconstruirse con los valores de `best_run.config` antes de cargar los pesos; de lo contrario `load_state_dict` fallará porque las formas de los tensores no coinciden.

In [ ]:
# restauramos el modelo en una instancia nueva (map_location permite cargar pesos guardados en otro dispositivo)
best_model = MLP(input_len).to(DEVICE)
best_model.load_state_dict(torch.load("model.pth", map_location=DEVICE))

# Evaluamos el modelo en el conjunto de test
test_loss = evaluate(best_model, CRITERION, test_loader)

print(f"Test Loss: {test_loss:.5f}")

## Ejercicios


1. **Expansión de la Configuración del Sweep**:
   - **Objetivo**: Ampliar la configuración del sweep existente para incluir la exploración de nuevas arquitecturas de red neuronal.
   - **Instrucción**: Expanda el `sweep_config` para incluir parámetros relacionados con la arquitectura del modelo, como el número de capas ocultas (`hidden_layers`) y el número de neuronas por capa (`neurons_per_layer`). Establezca un rango razonable para estos parámetros, por ejemplo, de 1 a 4 para las capas ocultas y de 16 a 128 para las neuronas por capa. Luego, ejecute el sweep y utilice W&B para analizar cómo estos cambios en la arquitectura afectan la métrica de pérdida de validación (`val_loss`).

2. **Experimentación con Funciones de Activación**:
   - **Objetivo**: Evaluar cómo diferentes funciones de activación afectan el rendimiento del modelo.
   - **Instrucción**: Añada una nueva variable a la configuración del sweep para probar diferentes funciones de activación (por ejemplo, ReLU, tanh, sigmoid). Modifique el `sweep_config` para incluir un parámetro `activation_function` con valores posibles como ["relu", "tanh", "sigmoid"]. Ejecute el sweep para explorar cuál función de activación resulta en un mejor rendimiento para el modelo y justifique los resultados basándose en las visualizaciones y métricas registradas en W&B.


```python
sweep_config = {
    "name": "sweep-california-housing-expanded-architecture",
    "method": "random",
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e-1},
        "optimizer": {"values": ["adam", "sgd"]},
        "batch_size": {"values": [32, 256, 1024, 4096]},
        "hidden_layers": {"values": [1, 2, 3, 4]},  # Número de capas ocultas
        "neurons_per_layer": {"values": [16, 32, 64, 128]},  # Número de neuronas por capa
        "activation_function": {"values": ["relu", "tanh", "sigmoid"]}  # Función de activación
    },
}
```

> **Nota:** al cambiar la arquitectura, `MLP` debe recibir estos hiperparámetros como argumentos y `sweep_run` debe construir el modelo a partir de `wandb.config`. En la evaluación final, reconstruya el modelo con `best_run.config` antes de llamar a `load_state_dict`.
